# **Red Neuronal de testing y resultados preliminares V.1**


## Llamar librerías


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

## 1) Cargar datos


In [ ]:
from pathlib import Path
DATOS_PROCESADOS = Path("data/procesados")
train_path = DATOS_PROCESADOS / "REGEN_basev6_X_train_ready.csv"
val_path   = DATOS_PROCESADOS / "REGEN_basev6_X_val_ready.csv"
test_path  = DATOS_PROCESADOS / "REGEN_basev6_X_test_ready.csv"

train_df = pd.read_csv(train_path, encoding="utf-8-sig")
val_df   = pd.read_csv(val_path, encoding="utf-8-sig")
test_df  = pd.read_csv(test_path, encoding="utf-8-sig")

y_train = train_df["y"].astype(int).values
X_train = train_df.drop(columns=["y"]).astype(np.float32).values

y_val = val_df["y"].astype(int).values
X_val = val_df.drop(columns=["y"]).astype(np.float32).values

y_test = test_df["y"].astype(int).values
X_test = test_df.drop(columns=["y"]).astype(np.float32).values

print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

## 2) class_weight (por desbalance)


In [ ]:
classes = np.array([0, 1])
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight = {0: float(weights[0]), 1: float(weights[1])}
print("class_weight:", class_weight)

## 3) Modelo MLP (pequeño/mediano)


In [ ]:
tf.random.set_seed(42)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),

    tf.keras.layers.Dense(256, kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.Dropout(0.35),

    tf.keras.layers.Dense(128, kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.Dropout(0.30),

    tf.keras.layers.Dense(64, kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.Dropout(0.20),

    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="acc"),
        tf.keras.metrics.AUC(name="roc_auc", curve="ROC"),
        tf.keras.metrics.AUC(name="pr_auc", curve="PR"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
    ]
)

model.summary()

## 4) Callbacks


In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_pr_auc", mode="max", patience=20,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_pr_auc", mode="max",
        factor=0.5, patience=8, min_lr=1e-6
    )
]

## 5) Entrenar


In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=32,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)

## 6) Evaluación en TEST


In [ ]:
test_metrics = model.evaluate(X_test, y_test, verbose=0)
print("\nTest metrics:", dict(zip(model.metrics_names, test_metrics)))

# Probabilidades y predicción con threshold 0.5 (luego podemos ajustar)
y_prob = model.predict(X_test).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred, digits=4))

print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC :", average_precision_score(y_test, y_prob))

---
## 7) Resultados preliminares consolidados







### 7.0 Encabezados de Variables usadas (dataset crudo)

In [ ]:
dataset_base_path = Path("data/Datos_depurados_base.xlsx")

dataset_base_df = pd.read_excel(dataset_base_path)

columnas_df = pd.DataFrame(dataset_base_df.columns, columns=["Columna"])

display(columnas_df)

### 7.1 Configuración de la ejecución preliminar

Se documenta la configuración utilizada en la evaluación para que los resultados publicados puedan vincularse posteriormente con esta ejecución específica.


In [ ]:
# ============================================================
# CONFIGURACIÓN Y CARPETAS DE RESULTADOS PRELIMINARES
# ============================================================
import os
import json
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    roc_curve,
    ConfusionMatrixDisplay
)

# Umbral utilizado en esta evaluación preliminar.
# Se conserva el mismo valor empleado en la evaluación original.
THRESHOLD_PRELIMINAR = 0.50

OUTPUT_ROOT = "outputs/resultados_mlp_v1"
PUBLIC_DIR = os.path.join(OUTPUT_ROOT, "graficos_publicacion")
TECH_DIR = os.path.join(OUTPUT_ROOT, "graficos_tecnicos")

os.makedirs(PUBLIC_DIR, exist_ok=True)
os.makedirs(TECH_DIR, exist_ok=True)

# Reutilizar probabilidades/predicciones si ya existen.
if "y_prob" not in globals():
    y_prob = model.predict(X_test, verbose=0).ravel()

# Para que toda esta sección use exactamente el mismo umbral explícito.
y_pred_preliminar = (y_prob >= THRESHOLD_PRELIMINAR).astype(int)

fecha_ejecucion = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

configuracion_ejecucion = {
    "tipo_resultado": "PRELIMINAR",
    "fecha_ejecucion": fecha_ejecucion,
    "modelo": "MLP - red neuronal multicapa",
    "threshold": THRESHOLD_PRELIMINAR,
    "n_test": int(len(y_test)),
    "n_permanencia_test": int(np.sum(np.asarray(y_test) == 0)),
    "n_desercion_test": int(np.sum(np.asarray(y_test) == 1)),
    "criterio_early_stopping": "val_pr_auc",
    "nota": (
        "Resultados de una única ejecución preliminar. "
        "No corresponden a la evaluación definitiva del modelo."
    )
}

display(pd.DataFrame([configuracion_ejecucion]))


### 7.2 Validaciones de coherencia

Antes de generar cualquier gráfico o texto de divulgación, se comprueba que probabilidades, predicciones, matriz de confusión y tamaño del conjunto de prueba sean consistentes entre sí.


In [ ]:
# ============================================================
# VALIDACIONES DE COHERENCIA
# ============================================================
y_test_np = np.asarray(y_test).astype(int).ravel()
y_prob_np = np.asarray(y_prob).astype(float).ravel()
y_pred_np = np.asarray(y_pred_preliminar).astype(int).ravel()

assert len(y_test_np) == len(y_prob_np), \
    "y_test y y_prob tienen longitudes distintas."

assert len(y_test_np) == len(y_pred_np), \
    "y_test y y_pred tienen longitudes distintas."

assert set(np.unique(y_test_np)).issubset({0, 1}), \
    "y_test contiene clases diferentes de 0 y 1."

assert np.all((y_prob_np >= 0) & (y_prob_np <= 1)), \
    "Existen probabilidades fuera del intervalo [0, 1]."

cm = confusion_matrix(y_test_np, y_pred_np, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

assert cm.sum() == len(y_test_np), \
    "La matriz de confusión no suma el total del conjunto TEST."

assert (tn + fp) == np.sum(y_test_np == 0), \
    "La fila de permanencia no coincide con el total real."

assert (fn + tp) == np.sum(y_test_np == 1), \
    "La fila de deserción no coincide con el total real."

print("Validaciones completadas correctamente.")
print(f"Total TEST: {len(y_test_np)}")
print(f"Permanencia real: {tn + fp}")
print(f"Deserción real: {fn + tp}")
print(f"Matriz de confusión: {cm.tolist()}")


### 7.3 Tabla única de métricas preliminares

Todas las visualizaciones y textos posteriores utilizan estas mismas variables como **fuente única de verdad**, evitando inconsistencias entre gráficos.


In [ ]:
# ============================================================
# MÉTRICAS PRELIMINARES - FUENTE ÚNICA DE VERDAD
# ============================================================
accuracy = accuracy_score(y_test_np, y_pred_np)
precision = precision_score(y_test_np, y_pred_np, zero_division=0)
recall = recall_score(y_test_np, y_pred_np, zero_division=0)
f1 = f1_score(y_test_np, y_pred_np, zero_division=0)

specificity = tn / (tn + fp) if (tn + fp) else np.nan
fnr = fn / (fn + tp) if (fn + tp) else np.nan

balanced_acc = balanced_accuracy_score(y_test_np, y_pred_np)
mcc = matthews_corrcoef(y_test_np, y_pred_np)
roc_auc = roc_auc_score(y_test_np, y_prob_np)
avg_precision = average_precision_score(y_test_np, y_prob_np)

metricas_preliminares = pd.DataFrame({
    "Métrica": [
        "Accuracy",
        "Precision - Deserción",
        "Recall / Sensibilidad - Deserción",
        "F1-score - Deserción",
        "Especificidad - Permanencia",
        "False Negative Rate (FNR)",
        "Balanced Accuracy",
        "Matthews Correlation Coefficient (MCC)",
        "ROC-AUC",
        "Average Precision"
    ],
    "Valor": [
        accuracy,
        precision,
        recall,
        f1,
        specificity,
        fnr,
        balanced_acc,
        mcc,
        roc_auc,
        avg_precision
    ]
})

display(metricas_preliminares.style.format({"Valor": "{:.4f}"}))

print("\nMatriz de confusión:")
print(cm)
print(f"\nCorrectos: {tn + tp} de {len(y_test_np)}")
print(f"Errores: {fp + fn} de {len(y_test_np)}")


### 7.4 Diagnóstico del entrenamiento

Se recuperan del objeto `history` los dos criterios de interés:

- época con menor pérdida de validación (`val_loss`);
- época con mayor PR-AUC de validación (`val_pr_auc`).

La coincidencia o diferencia entre ambos criterios se documenta automáticamente.


In [ ]:
# ============================================================
# DIAGNÓSTICO DEL ENTRENAMIENTO
# ============================================================
if "history" not in globals():
    raise RuntimeError(
        "No se encontró `history`. Ejecuta primero la celda de model.fit(...)."
    )

for key in ["loss", "val_loss", "pr_auc", "val_pr_auc"]:
    if key not in history.history:
        raise KeyError(
            f"No se encontró '{key}' en history.history."
        )

train_loss = history.history["loss"]
val_loss = history.history["val_loss"]
train_pr_auc = history.history["pr_auc"]
val_pr_auc = history.history["val_pr_auc"]

best_val_loss_epoch = int(np.argmin(val_loss)) + 1
best_val_loss = float(np.min(val_loss))

best_pr_epoch = int(np.argmax(val_pr_auc)) + 1
best_val_pr_auc = float(np.max(val_pr_auc))

diagnostico_entrenamiento = pd.DataFrame({
    "Indicador": [
        "Épocas ejecutadas",
        "Mejor época según val_loss",
        "val_loss mínima",
        "Mejor época según val_PR-AUC",
        "val_PR-AUC máxima",
        "Coincidencia de criterios"
    ],
    "Valor": [
        len(train_loss),
        best_val_loss_epoch,
        best_val_loss,
        best_pr_epoch,
        best_val_pr_auc,
        "Sí" if best_val_loss_epoch == best_pr_epoch else "No"
    ]
})

display(diagnostico_entrenamiento)

if best_val_loss_epoch == best_pr_epoch:
    print(
        f"Ambos criterios de validación alcanzaron su mejor valor "
        f"en la época {best_pr_epoch}."
    )
else:
    print(
        f"Los criterios difieren: val_loss fue mínima en la época "
        f"{best_val_loss_epoch} y val_PR-AUC fue máxima en la época "
        f"{best_pr_epoch}."
    )


## 7.5 Gráficos para divulgación institucional


### 7.5.1 Exactitud global


In [ ]:
# ============================================================
# GRÁFICO PÚBLICO 1 - TARJETA DE ACCURACY
# ============================================================
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.axis("off")

ax.text(
    0.5, 0.67,
    f"{accuracy:.1%}",
    ha="center",
    va="center",
    fontsize=42,
    fontweight="bold",
    transform=ax.transAxes
)

ax.text(
    0.5, 0.47,
    "Exactitud global del modelo",
    ha="center",
    va="center",
    fontsize=18,
    fontweight="bold",
    transform=ax.transAxes
)

ax.text(
    0.5, 0.31,
    f"{tn + tp} de {len(y_test_np)} casos correctamente clasificados",
    ha="center",
    va="center",
    fontsize=13,
    transform=ax.transAxes
)

ax.text(
    0.5, 0.10,
    "Resultado preliminar — una ejecución sobre el conjunto de prueba",
    ha="center",
    va="center",
    fontsize=10,
    transform=ax.transAxes
)

fig.tight_layout()

path_accuracy = os.path.join(PUBLIC_DIR, "01_accuracy_preliminar.png")
fig.savefig(path_accuracy, dpi=300, bbox_inches="tight")
plt.show()


### 7.5.2 Curva ROC


In [ ]:
# ============================================================
# GRÁFICO PÚBLICO 2 - CURVA ROC
# ============================================================
fpr, tpr, _ = roc_curve(y_test_np, y_prob_np)

fig, ax = plt.subplots(figsize=(7.5, 6))

ax.plot(
    fpr,
    tpr,
    linewidth=2.3,
    label=f"Modelo MLP (AUC = {roc_auc:.4f})"
)

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.5,
    label="Clasificador aleatorio"
)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.set_xlabel("Tasa de falsos positivos")
ax.set_ylabel("Tasa de verdaderos positivos (sensibilidad)")
ax.set_title(
    "Capacidad de discriminación del modelo",
    fontsize=15,
    fontweight="bold",
    pad=14
)

ax.legend(loc="lower right")
ax.grid(alpha=0.2)

fig.text(
    0.5, 0.01,
    "Resultado preliminar — una ejecución sobre el conjunto de prueba",
    ha="center",
    fontsize=9
)

fig.tight_layout(rect=[0, 0.04, 1, 1])

path_roc_public = os.path.join(PUBLIC_DIR, "02_curva_roc_preliminar.png")
fig.savefig(path_roc_public, dpi=300, bbox_inches="tight")
plt.show()


### 7.5.3 Matriz de confusión normalizada


In [ ]:
# ============================================================
# GRÁFICO PÚBLICO 3 - MATRIZ NORMALIZADA
# ============================================================
cm_norm = confusion_matrix(
    y_test_np,
    y_pred_np,
    labels=[0, 1],
    normalize="true"
)

fig, ax = plt.subplots(figsize=(7, 6))

disp_norm = ConfusionMatrixDisplay(
    confusion_matrix=cm_norm,
    display_labels=["Permanencia", "Deserción"]
)

disp_norm.plot(
    ax=ax,
    values_format=".1%",
    colorbar=False
)

ax.set_xlabel("Predicción del modelo")
ax.set_ylabel("Condición real")
ax.set_title(
    "Resultados de clasificación por condición real",
    fontsize=15,
    fontweight="bold",
    pad=14
)

fig.text(
    0.5, 0.01,
    "Resultado preliminar — normalización por condición real",
    ha="center",
    fontsize=9
)

fig.tight_layout(rect=[0, 0.04, 1, 1])

path_cm_norm_public = os.path.join(
    PUBLIC_DIR,
    "03_matriz_confusion_normalizada_preliminar.png"
)

fig.savefig(path_cm_norm_public, dpi=300, bbox_inches="tight")
plt.show()


### 7.5.4 Curva de pérdida


In [ ]:
# ============================================================
# GRÁFICO PÚBLICO 4 - CURVA DE PÉRDIDA
# ============================================================
epochs_loss = range(1, len(train_loss) + 1)

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(
    epochs_loss,
    train_loss,
    label="Entrenamiento",
    linewidth=2.2
)

ax.plot(
    epochs_loss,
    val_loss,
    label="Validación",
    linewidth=2.2
)

ax.axvline(
    best_val_loss_epoch,
    linestyle="--",
    linewidth=1.4,
    alpha=0.7,
    label=f"Menor pérdida de validación (época {best_val_loss_epoch})"
)

ax.scatter(
    [best_val_loss_epoch],
    [best_val_loss],
    s=55,
    zorder=3
)

ax.set_xlabel("Época")
ax.set_ylabel("Pérdida")
ax.set_title(
    "Evolución del error durante el entrenamiento",
    fontsize=15,
    fontweight="bold",
    pad=14
)

ax.legend()
ax.grid(alpha=0.2)

fig.text(
    0.5, 0.01,
    "Resultado preliminar — entrenamiento y validación",
    ha="center",
    fontsize=9
)

fig.tight_layout(rect=[0, 0.04, 1, 1])

path_loss_public = os.path.join(
    PUBLIC_DIR,
    "04_curva_perdida_preliminar.png"
)

fig.savefig(path_loss_public, dpi=300, bbox_inches="tight")
plt.show()


## 7.6 Gráficos técnicos para TFG


### 7.6.1 Matriz de confusión absoluta


In [ ]:
# ============================================================
# GRÁFICO TÉCNICO 1 - MATRIZ ABSOLUTA
# ============================================================
fig, ax = plt.subplots(figsize=(7, 6))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Permanencia", "Deserción"]
)

disp.plot(
    ax=ax,
    values_format="d",
    colorbar=False
)

ax.set_xlabel("Predicción del modelo")
ax.set_ylabel("Condición real")
ax.set_title(
    "Matriz de confusión del modelo",
    fontsize=14,
    fontweight="bold",
    pad=14
)

fig.tight_layout()

path_cm_abs = os.path.join(
    TECH_DIR,
    "01_matriz_confusion_absoluta.png"
)

fig.savefig(path_cm_abs, dpi=300, bbox_inches="tight")
plt.show()


### 7.6.2 Evolución del PR-AUC


In [ ]:
# ============================================================
# GRÁFICO TÉCNICO 2 - PR-AUC TRAIN VS VALIDATION
# ============================================================
epochs_pr = range(1, len(train_pr_auc) + 1)

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(
    epochs_pr,
    train_pr_auc,
    label="Entrenamiento",
    linewidth=2.2
)

ax.plot(
    epochs_pr,
    val_pr_auc,
    label="Validación",
    linewidth=2.2
)

ax.axvline(
    best_pr_epoch,
    linestyle="--",
    linewidth=1.4,
    alpha=0.7,
    label=f"Mejor val_PR-AUC (época {best_pr_epoch})"
)

ax.scatter(
    [best_pr_epoch],
    [best_val_pr_auc],
    s=55,
    zorder=3
)

ax.annotate(
    f"val_PR-AUC máxima = {best_val_pr_auc:.4f}",
    xy=(best_pr_epoch, best_val_pr_auc),
    xytext=(10, -28),
    textcoords="offset points",
    fontsize=10
)

ax.set_xlabel("Época")
ax.set_ylabel("PR-AUC")
ax.set_title(
    "Evolución del PR-AUC durante el entrenamiento",
    fontsize=14,
    fontweight="bold",
    pad=14
)

ax.set_ylim(0, 1.02)
ax.legend()
ax.grid(alpha=0.2)

fig.tight_layout()

path_pr_auc = os.path.join(
    TECH_DIR,
    "02_curva_pr_auc_entrenamiento.png"
)

fig.savefig(path_pr_auc, dpi=300, bbox_inches="tight")
plt.show()


### 7.6.3 Matriz de confusión normalizada para revisión técnica


In [ ]:
# ============================================================
# GRÁFICO TÉCNICO 3 - MATRIZ NORMALIZADA
# ============================================================
fig, ax = plt.subplots(figsize=(7, 6))

disp_norm_tech = ConfusionMatrixDisplay(
    confusion_matrix=cm_norm,
    display_labels=["Permanencia", "Deserción"]
)

disp_norm_tech.plot(
    ax=ax,
    values_format=".1%",
    colorbar=False
)

ax.set_xlabel("Predicción del modelo")
ax.set_ylabel("Condición real")
ax.set_title(
    "Matriz de confusión normalizada",
    fontsize=14,
    fontweight="bold",
    pad=14
)

fig.tight_layout()

path_cm_norm_tech = os.path.join(
    TECH_DIR,
    "03_matriz_confusion_normalizada.png"
)

fig.savefig(path_cm_norm_tech, dpi=300, bbox_inches="tight")
plt.show()


## 7.7 Interpretaciones automáticas


In [ ]:
# ============================================================
# INTERPRETACIÓN TÉCNICA Y DIVULGATIVA
# ============================================================
texto_tecnico = (
    f"En esta ejecución preliminar, el modelo alcanzó un accuracy de "
    f"{accuracy:.2%}, una precisión para la clase deserción de "
    f"{precision:.2%}, un recall de {recall:.2%}, un F1-score de "
    f"{f1:.2%} y un ROC-AUC de {roc_auc:.4f}. La matriz de confusión "
    f"registró {tn} verdaderos negativos, {fp} falsos positivos, "
    f"{fn} falsos negativos y {tp} verdaderos positivos. "
    f"Durante el entrenamiento, la menor pérdida de validación se obtuvo "
    f"en la época {best_val_loss_epoch} (val_loss={best_val_loss:.4f}), "
    f"mientras que el mayor PR-AUC de validación se obtuvo en la época "
    f"{best_pr_epoch} (val_PR-AUC={best_val_pr_auc:.4f}). "
    f"Estos resultados corresponden a una única ejecución preliminar y "
    f"no constituyen todavía la evaluación definitiva del modelo."
)

texto_divulgativo = (
    f"En una evaluación preliminar sobre el conjunto de prueba, el modelo "
    f"clasificó correctamente {tn + tp} de los {len(y_test_np)} casos "
    f"analizados, equivalente a una exactitud global del {accuracy:.1%}. "
    f"De los {tp + fn} casos reales de deserción, identificó correctamente "
    f"{tp}, mientras que {fn} no fueron detectados. La curva ROC alcanzó "
    f"un área bajo la curva de {roc_auc:.4f}, evidenciando una alta "
    f"capacidad preliminar para diferenciar los patrones asociados con "
    f"permanencia y deserción. Estos resultados son preliminares y serán "
    f"complementados posteriormente con evaluaciones adicionales de "
    f"estabilidad y generalización."
)

print("INTERPRETACIÓN TÉCNICA\n")
print(texto_tecnico)

print("\n" + "=" * 80 + "\n")

print("INTERPRETACIÓN PARA DIVULGACIÓN\n")
print(texto_divulgativo)


## 7.8 Exportación y registro de la ejecución

Además de los gráficos, se guarda un registro estructurado de la ejecución preliminar para poder identificar exactamente qué resultados fueron compartidos con el tutor o utilizados en divulgación institucional.


In [ ]:
# ============================================================
# EXPORTACIÓN CONSOLIDADA
# ============================================================
metricas_csv = os.path.join(
    OUTPUT_ROOT,
    "metricas_preliminares.csv"
)

metricas_preliminares.to_csv(
    metricas_csv,
    index=False,
    encoding="utf-8-sig"
)

diagnostico_csv = os.path.join(
    OUTPUT_ROOT,
    "diagnostico_entrenamiento.csv"
)

diagnostico_entrenamiento.to_csv(
    diagnostico_csv,
    index=False,
    encoding="utf-8-sig"
)

texto_tecnico_path = os.path.join(
    OUTPUT_ROOT,
    "interpretacion_tecnica.txt"
)

with open(texto_tecnico_path, "w", encoding="utf-8") as f:
    f.write(texto_tecnico)

texto_publico_path = os.path.join(
    OUTPUT_ROOT,
    "interpretacion_divulgativa.txt"
)

with open(texto_publico_path, "w", encoding="utf-8") as f:
    f.write(texto_divulgativo)

snapshot = {
    **configuracion_ejecucion,
    "metricas": {
        "accuracy": float(accuracy),
        "precision_desercion": float(precision),
        "recall_desercion": float(recall),
        "f1_desercion": float(f1),
        "specificity": float(specificity),
        "false_negative_rate": float(fnr),
        "balanced_accuracy": float(balanced_acc),
        "mcc": float(mcc),
        "roc_auc": float(roc_auc),
        "average_precision": float(avg_precision)
    },
    "matriz_confusion": {
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "matrix": cm.tolist()
    },
    "entrenamiento": {
        "epocas_ejecutadas": int(len(train_loss)),
        "mejor_epoca_val_loss": int(best_val_loss_epoch),
        "val_loss_minima": float(best_val_loss),
        "mejor_epoca_val_pr_auc": int(best_pr_epoch),
        "val_pr_auc_maxima": float(best_val_pr_auc),
        "coincidencia_criterios": bool(
            best_val_loss_epoch == best_pr_epoch
        )
    }
}

snapshot_path = os.path.join(
    OUTPUT_ROOT,
    "snapshot_resultados_preliminares.json"
)

with open(snapshot_path, "w", encoding="utf-8") as f:
    json.dump(snapshot, f, ensure_ascii=False, indent=2)

print("Exportación completada.\n")
print("Carpeta raíz:", OUTPUT_ROOT)
print("Gráficos para publicación:", PUBLIC_DIR)
print("Gráficos técnicos:", TECH_DIR)
print("Métricas:", metricas_csv)
print("Diagnóstico:", diagnostico_csv)
print("Snapshot:", snapshot_path)


## 8) Guardar modelo


In [ ]:
#model.save("modelo_desercion_mlp.keras")
#print("\nModelo guardado en: modelo_desercion_mlp.keras")